# Germline Whole Exome Sequencing (WES): Clinical Variant Interpretation

**Author:** Eman Koosehlar

**Notebook:** 3 of the series  

**Version:** 1.0

**Last Updated:** July 2026

**License:** MIT License

---

### Runtime Requirements

> <font color = 'yellow'>  This notebook requires a Google Colab runtime with sufficient temporary storage to accommodate the Exomiser database and supporting reference files.</font>

**Recommended runtime**
- Runtime type: `TPU`

- Storage: Ensure sufficient free disk space before downloading the required resources.

> **Note:** A TPU is required for Exomiser to increase available storage.

---

---

### Table of contents<a id='toc0_'></a>    


- [Introduction](#toc1_)
- [Requirements and Environment Setup](#toc2_)
- [Workflow Overview](#toc3_)
- [Clinical Case 01](#toc4_)
  - [Clinical Presentation](#toc4_1_)
  - [Load the Input VCF](#toc4_2_)
  - [Patient Phenotype](#toc4_3_)
  - [Phenotype-driven Prioritization](#toc4_4_)
      - [Required Inputs](#to4_4_1_)
      - [Exomiser Workflow](#to4_4_2_)
      - [Exomiser Configuration](#to4_4_3_)
      - [Run Exomiser](#to4_4_4_)

  - [Candidate Variant Review](#toc4_5_)
  - [Gene Review](#toc4_6_)
  - [Evidence Assessment (ACMG/AMP)](#toc4_7_)
  - [Variant Classification](#toc4_8_)
  - [Clinical Interpretation Report](#toc4_9_)


- [Summary](#toc5_)
- [References](#toc6_)


---

## <a id='toc1_'></a>[Introduction](#toc0_)

### Learning Journey

This notebook is the third part of the **Germline Whole Exome Sequencing (WES) Analysis Series**.

In the previous notebooks, we learned how to generate a high-confidence germline VCF through variant calling and quality filtering, and how to annotate and prioritize variants using biological and population-based evidence.

In this notebook, we move beyond variant discovery and begin the process of **clinical variant interpretation**.

Rather than focusing on individual software tools or guidelines, this notebook demonstrates how different sources of evidence are integrated to identify the variant most likely responsible for a patient's disease. We will combine genomic data with clinical information, phenotype descriptions, inheritance patterns, and selected ACMG/AMP criteria to support evidence-based variant classification.

To make the learning experience more realistic, this notebook is organized as a growing collection of simulated clinical cases. Each case represents a different interpretation scenario that introduces new concepts, workflows, and challenges commonly encountered in clinical genomics.

---

### Learning Objectives

By the end of this notebook, you will be able to:

* Understand the overall workflow of clinical variant interpretation.
* Convert clinical findings into Human Phenotype Ontology (HPO) terms.
* Perform phenotype-driven variant prioritization using Exomiser.
* Evaluate candidate variants by integrating phenotype, inheritance, population frequency, computational predictions, and gene–disease associations.
* Apply selected ACMG/AMP evidence criteria to support variant classification.
* Explain the reasoning behind identifying a candidate variant as the most likely cause of the patient's phenotype.
* Produce a structured clinical interpretation suitable for downstream reporting.

---

### Clinical Case Library

This notebook is designed as a collection of independent clinical interpretation exercises. New cases will be added over time to demonstrate different inheritance patterns, disease mechanisms, and interpretation strategies.

#### Current Cases

| Case | Clinical Scenario | Status | Notebook 3 |
|------|-------------------|--------|------------|
| Case 01 | Congenital craniofacial disorder | ✅ Available | ✅ Interpretation available |
| Case 02 | Autosomal recessive disorder | 🚧 Planned | 🚧 Planned |
| Case 03 | 🚧 Planned | 🚧 Planned | 🚧 Planned |
| Case 04 | 🚧 Planned | 🚧 Planned | 🚧 Planned |
| Case 05 | 🚧 Planned | 🚧 Planned | 🚧 Planned |

Each case begins with a set of candidate variants and limited clinical information. Your task is to evaluate the available evidence and determine which variant is most likely responsible for the patient's condition.

---

### Overview

This notebook begins with the prioritized candidate variants generated in the previous notebook ([**Germline WES Variant Annotation**](https://github.com/namia47/WES_Analysis_Tutorial/notebooks/02_Germline_Variant_Annotation.ipynb)) and follows the next stage of the germline WES analysis workflow: **clinical variant interpretation**.

For each clinical case, we will:

1. Review the patient's clinical presentation.
2. Define phenotype terms using the Human Phenotype Ontology (HPO).
3. Prioritize candidate variants using phenotype-driven analysis.
4. Investigate the highest-ranked candidate genes.
5. Collect supporting evidence from multiple sources.
6. Apply selected ACMG/AMP criteria.
7. Classify the candidate variant and explain the clinical reasoning behind the final interpretation.

Unlike the previous notebooks, there is no single correct command that identifies the causal variant. Clinical interpretation is an evidence-based reasoning process that integrates genomic data with clinical knowledge to support informed conclusions.

> **Educational Note:** The clinical cases presented in this notebook are simulated for educational purposes. Although the workflow reflects commonly used practices in clinical genomics, the examples should not be used for clinical diagnosis or medical decision-making.


---

## <a id='toc2_'></a>[Requirements and Environment Setup](#toc0_)

In [ ]:
from google.colab import auth
auth.authenticate_user()

**Installation Exomiser tool and related files** 

We use the hg19 version here, but this can be done with hg38. 

In [ ]:
# download the distribution
!wget https://data.monarchinitiative.org/exomiser/latest/exomiser-cli-14.0.0-distribution.zip

# Download the data for hg19 genome (this is ~80GB and will take about 5-10 minutes).
!wget https://data.monarchinitiative.org/exomiser/latest/2406_hg19.zip
!wget https://data.monarchinitiative.org/exomiser/latest/2406_phenotype.zip


--2025-07-16 15:24:02--  https://data.monarchinitiative.org/exomiser/latest/exomiser-cli-14.0.0-distribution.zip
Resolving data.monarchinitiative.org (data.monarchinitiative.org)... 35.208.191.193
Connecting to data.monarchinitiative.org (data.monarchinitiative.org)|35.208.191.193|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 75878204 (72M) [application/zip]
Saving to: ‘exomiser-cli-14.0.0-distribution.zip’

exomiser-cli-14.0.0 100%[===================>]  72.36M   107MB/s    in 0.7s    

2025-07-16 15:24:03 (107 MB/s) - ‘exomiser-cli-14.0.0-distribution.zip’ saved [75878204/75878204]

--2025-07-16 15:24:03--  https://data.monarchinitiative.org/exomiser/latest/2406_hg19.zip
Resolving data.monarchinitiative.org (data.monarchinitiative.org)... 35.208.191.193
Connecting to data.monarchinitiative.org (data.monarchinitiative.org)|35.208.191.193|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19329398823 (18G) [application/zip]
Saving to: 

- Extracting the distribution and data files; this will create a directory called 'exomiser-cli-14.0.0' in the current working directory.
- Moving data files to the exomiser-cli-14.0.0//data directory

In [ ]:
!unzip exomiser-cli-14.0.0-distribution.zip
!unzip 2406_hg19.zip -d exomiser-cli-14.0.0//data
!unzip 2406_phenotype.zip -d exomiser-cli-14.0.0/data

Archive:  exomiser-cli-14.0.0-distribution.zip
   creating: exomiser-cli-14.0.0/
   creating: exomiser-cli-14.0.0/examples/
   creating: exomiser-cli-14.0.0/lib/
  inflating: exomiser-cli-14.0.0/CHANGELOG.md  
  inflating: exomiser-cli-14.0.0/application.properties  
  inflating: exomiser-cli-14.0.0/examples/NA19722_601952_AUTOSOMAL_RECESSIVE_POMP_13_29233225_5UTR_38.yml  
  inflating: exomiser-cli-14.0.0/examples/output-options.yml  
  inflating: exomiser-cli-14.0.0/examples/preset-genome-analysis.yml  
  inflating: exomiser-cli-14.0.0/examples/test-analysis-batch.txt  
  inflating: exomiser-cli-14.0.0/examples/test-analysis-genome.yml  
  inflating: exomiser-cli-14.0.0/examples/exome-analysis.yml  
  inflating: exomiser-cli-14.0.0/examples/pfeiffer-family.yml  
  inflating: exomiser-cli-14.0.0/examples/pfeiffer-output-options.yml  
  inflating: exomiser-cli-14.0.0/examples/pfeiffer-phenopacket.yml  
  inflating: exomiser-cli-14.0.0/examples/NA19722_252900_AR_SGSH_1_NONSYNONYMOUS.vcf 

In [ ]:
# Removing unused ziped files
!rm 2406_hg19.zip
!rm exomiser-cli-14.0.0-distribution.zip
!rm 2406_phenotype.zip

In [ ]:
#installing Java 17 required for Exomiser
!apt-get install openjdk-17-jdk -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-dejavu-core fonts-dejavu-extra libatk-wrapper-java
  libatk-wrapper-java-jni libxt-dev libxtst6 libxxf86dga1
  openjdk-17-jdk-headless openjdk-17-jre openjdk-17-jre-headless x11-utils
Suggested packages:
  libxt-doc openjdk-17-demo openjdk-17-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  fonts-dejavu-core fonts-dejavu-extra libatk-wrapper-java
  libatk-wrapper-java-jni libxt-dev libxtst6 libxxf86dga1 openjdk-17-jdk
  openjdk-17-jdk-headless openjdk-17-jre openjdk-17-jre-headless x11-utils
0 upgraded, 12 newly installed, 0 to remove and 1 not upgraded.
Need to get 126 MB of archives.
After this operation, 288 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/ma

- **Project Directories**

To keep the workflow organized and reproducible, all files generated during the analysis are stored in dedicated project directories.

```text
WES_Analysis/
│
├── Annotation/
│   └── Annotated VCF files
│
├── Exomiser/
│   └── Exomiser outputs(VCF, TSV, htlm)
│
└── Configs/
    └── Exomiser configs fils(yml, properties)
```


In [ ]:
!mkdir /content/WES_Analysis
!mkdir /content/WES_Analysis/Annotation /content/WES_Analysis/Exomiser /content/WES_Analysis/Configs


---

## <a id='toc3_'></a>[Workflow Overview](#toc0_)

### Workflow Overview

Unlike the previous notebooks, this workflow focuses on **clinical reasoning** rather than data processing.

Starting from an annotated VCF containing thousands of variants, we will combine genomic data with clinical information to progressively narrow the search for the most likely disease-causing variant.

For each clinical case, we will:

1. Clinical presentation fo cases
2. Import the vcf files required for clinical interpretation.
3. Define the patient's phenotype using Human Phenotype Ontology (HPO) terms.
4. Prioritize variants using phenotype-driven analysis with Exomiser.
5. Review the highest-ranked candidate variants.
6. Investigate the most relevant candidate gene.
7. Collect supporting evidence using selected ACMG/AMP criteria.
8. Variant clasification based on the criteria.
9. Produce a final clinical interpretation based on the available evidence.
10. Case summary 

This workflow demonstrates how bioinformatics results are transformed into clinically meaningful conclusions through systematic evidence-based reasoning.


---

## <a id='toc4_'></a>[Clinical Case 01](#toc0_)

### <a id='toc4_1_'></a>[Clinical Presentation](#toc0_)

A child was referred for genetic evaluation because of multiple congenital craniofacial and skeletal abnormalities observed since birth.

Physical examination revealed *abnormal head shape with premature fusion of the cranial sutures (craniosynostosis), midface hypoplasia, broad and medially deviated thumbs and great toes, and partial soft tissue syndactyly involving the hands and feet*. Developmental milestones were appropriate for age, and no major internal organ abnormalities were identified.

There was no known family history of a similar condition. The clinical geneticist suspected an underlying monogenic disorder affecting craniofacial and skeletal development and requested Whole Exome Sequencing (WES) to identify a potential disease-causing variant.

The following sections will guide the interpretation of this case by integrating genomic findings with the patient's clinical features to identify the most likely molecular diagnosis.


---

### <a id='toc4_2_'></a>[Load the Input VCF](#toc0_)

Before beginning the interpretation, we import the files required for the analysis include the annotated VCF generated in the previous notebook.


- **Option A (*Recommended*):** Use your own annotated VCF

Use the annotated VCF If you completed the Notebook 2.

This provides the best learning experience and demonstrates the complete WES analysis workflow.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

!cp /content/gdrive/MyDrive/WES_Analysis/Annotation/clinical_case_01_annotated.vcf /content/WES_Analysis_Tutorial/Annotation
Clinical_Case_01 = "/content/WES_Analysis_Tutorial/Annotation/clinical_case_01_annotated.vcf"

-  **Option B:** Download the prepared example

If you want to start directly from this notebook, you may download the prepared example dataset from the project GitHub.

In [ ]:
#!git clone https://github.com/namia47/WES_Analysis_Tutorial.git

In [ ]:
#Clinical_Case_01_Annotated = "/content/WES_Analysis_Tutorial/data/clinical_case_01_annotated.vcf"

---

### <a id='toc4_3_'></a>[Patient Phenotype](#toc0_)

To perform phenotype-driven variant prioritization, the clinical findings are converted into standardized Human Phenotype Ontology (HPO) terms.

| Clinical finding | HPO term | HPO ID |
|------------------|----------|--------|
| Craniosynostosis | Craniosynostosis | HP:0001363 |
| Midface hypoplasia | Midface retrusion | HP:0011800 |
| Broad thumb | Broad thumb | HP:0011304 |
| Broad great toe | Broad hallux | HP:0010055 |
| Brachydactyly | Brachydactyly | HP:0001156 |

---

### <a id='toc4_4_'></a>[Phenotype-driven Prioritization](#toc0_)

At this stage, we have an annotated VCF containing thousands of genetic variants. Although each variant has been enriched with biological information, manually reviewing every variant is impractical.

To focus the analysis, we combine the patient's phenotype with the annotated variant dataset using **Exomiser**, a phenotype-driven prioritization tool. Exomiser integrates variant information, gene–disease associations, inheritance models, and Human Phenotype Ontology (HPO) terms to rank candidate genes and variants according to their likelihood of explaining the patient's clinical presentation.

Rather than classifying variants according to the ACMG/AMP guidelines, the objective of this step is to identify a small number of high-priority candidate variants that warrant detailed clinical review in the following sections.


---

#### <a id='toc4_4_1_'></a>[Required Inputs](#toc0_)


Exomiser requires the following input files:

| Input | Purpose |
|--------|---------|
| Annotated VCF | Contains the patient's genetic variants |
| HPO terms | Describe the patient's phenotype |
| Exomiser configuration (.yml) | Defines the analysis settings |
| Properties file (application.properties) | Defines the global settings |
| Reference databases | Gene, phenotype, and disease annotations used by Exomiser |

#### <a id='toc4_4_2_'></a>[Exomiser Workflow](#toc0_)

Exomiser evaluates both the predicted impact of each variant and how well the associated gene matches the patient's phenotype.

<img src ="Workflow_Exomiser.png">

#### <a id='toc4_4_3_'></a>[Exomiser Configuration](#toc0_)

Before running Exomiser, we need to specify the input files and analysis settings. These settings are stored in two configuration files:

* **YAML configuration file (`.yml`)** – defines the analysis for the current patient or clinical case.
* **Properties file (`application.properties`)** – defines the location of the Exomiser reference databases and other global settings.

Separating these files allows the same Exomiser installation to be reused for multiple clinical cases by simply creating a new YAML configuration for each analysis.


**YAML Configuration File**

The YAML configuration file contains the information required to analyze an individual clinical case. Rather than modifying the analysis command each time, we define the input files and analysis parameters in a single configuration file.

For this notebook, the most important settings are:

| Parameter                     | Purpose                                                                                                               |
| ----------------------------- | --------------------------------------------------------------------------------------------------------------------- |
| Genome assembly               | Specifies the reference genome used by the input VCF (e.g., GRCh37 or GRCh38).                                        |
| VCF path                      | Path to the annotated VCF generated in Notebook 2.                                                                    |
| HPO terms                     | Standardized patient phenotypes used for phenotype-driven prioritization.                                             |
| Frequency threshould          | Filters the common variants based on population data.                                             |
| Output directory              | Location where Exomiser will save the analysis results.                                                               |
| Inheritance mode *(if applicable)* | Restricts the analysis to one or more expected inheritance models, such as autosomal dominant or autosomal recessive. |

Additional parameters are available, but the defaults are sufficient for this educational workflow.


In [ ]:
%%writefile /content/WES_Analysis/Configs/Clinical_Case_01.yml


## Exomiser Analysis Template.
# These are all the possible options for running exomiser. Use this as a template for
# your own set-up.
---
analysis:
    # hg19 or hg38 - ensure that the application has been configured to run the specified assembly otherwise it will halt.
    genomeAssembly: hg19
    vcf: {Clinical_Case_01} #Clinical_Case_01 = "/content/gdrive/MyDrive/WES_Analysis/Annotation/clinical_case_01_annotated.vcf"
    ped:
    proband:
    hpoIds: ['HP:0001156', # Brachydactyly
             'HP:0001363', # Craniosynostosis
             'HP:0011304', # Broad thumb
             'HP:0010055'  # Broad hallux
             ]
    # These are the default settings, with values representing the maximum minor allele frequency in percent (%) permitted for an
    # allele to be considered as a causative candidate under that mode of inheritance.
    # If you just want to analyse a sample under a single inheritance mode, delete/comment-out the others. For AUTOSOMAL_RECESSIVE
    # or X_RECESSIVE ensure *both* relevant HOM_ALT and COMP_HET modes are present.
    # In cases where you do not want any cut-offs applied an empty map should be used e.g. inheritanceModes: {}
    inheritanceModes: {
      AUTOSOMAL_DOMINANT: 0.1,
      AUTOSOMAL_RECESSIVE_HOM_ALT: 0.1,
      AUTOSOMAL_RECESSIVE_COMP_HET: 2.0,
      X_DOMINANT: 0.1,
      X_RECESSIVE_HOM_ALT: 0.1,
      X_RECESSIVE_COMP_HET: 2.0,
      MITOCHONDRIAL: 0.2
    }
  #FULL or PASS_ONLY
    analysisMode: PASS_ONLY
  # Possible frequencySources:
  # UK10K - http://www.uk10k.org/ (UK10K)
  # gnomAD - http://gnomad.broadinstitute.org/ (GNOMAD_E, GNOMAD_G)
  # note that as of gnomAD v2.1 1000 genomes, ExAC are part of gnomAD
  # as of gnomAD v4 TOPMed & ESP are also included in gnomAD
    frequencySources: [
        UK10K,

        GNOMAD_E_AFR,
        GNOMAD_E_AMR,
      #  GNOMAD_E_ASJ,
        GNOMAD_E_EAS,
      #  GNOMAD_E_FIN,
        GNOMAD_E_NFE,
      #  GNOMAD_E_OTH,
        GNOMAD_E_SAS,

        GNOMAD_G_AFR,
        GNOMAD_G_AMR,
      #  GNOMAD_G_ASJ,
        GNOMAD_G_EAS,
      #  GNOMAD_G_FIN,
        GNOMAD_G_NFE,
      #  GNOMAD_G_OTH,
        GNOMAD_G_SAS
    ]
  # Possible pathogenicitySources: (POLYPHEN, MUTATION_TASTER, SIFT), (REVEL, MVP), CADD, REMM, SPLICE_AI, ALPHA_MISSENSE
  # REMM is trained on non-coding regulatory regions
  # *WARNING* if you enable CADD or REMM ensure that you have downloaded and installed the CADD/REMM tabix files
  # and updated their location in the application.properties. Exomiser will not run without this.
    pathogenicitySources: [ REVEL, MVP ]
  # this is the standard exomiser order.
  # all steps are optional
    steps: [
      #intervalFilter: {interval: 'chr10:123256200-123256300'},
      # or for multiple intervals:
      #intervalFilter: {intervals: ['chr10:123256200-123256300', 'chr10:123256290-123256350']},
      # or using a BED file - NOTE this should be 0-based, Exomiser otherwise uses 1-based coordinates in line with VCF
      #intervalFilter: {bed: /full/path/to/bed_file.bed},
      #genePanelFilter: {geneSymbols: ['FGFR1','FGFR2']},
      # geneBlacklistFilter: { },
        failedVariantFilter: { },
      #qualityFilter: {minQuality: 50.0},
        variantEffectFilter: {
          remove: [
              FIVE_PRIME_UTR_EXON_VARIANT,
              FIVE_PRIME_UTR_INTRON_VARIANT,
              THREE_PRIME_UTR_EXON_VARIANT,
              THREE_PRIME_UTR_INTRON_VARIANT,
              NON_CODING_TRANSCRIPT_EXON_VARIANT,
              NON_CODING_TRANSCRIPT_INTRON_VARIANT,
              CODING_TRANSCRIPT_INTRON_VARIANT,
                UPSTREAM_GENE_VARIANT,
                DOWNSTREAM_GENE_VARIANT,
                INTERGENIC_VARIANT,
                REGULATORY_REGION_VARIANT
            ]
        },
        #knownVariantFilter: {}, #removes variants represented in the database
        frequencyFilter: {maxFrequency: 2.0},
        pathogenicityFilter: {keepNonPathogenic: true},
        #inheritanceFilter and omimPrioritiser should always run AFTER all other filters have completed
        #they will analyse genes according to the specified modeOfInheritance above- UNDEFINED will not be analysed.
        inheritanceFilter: {},
        #omimPrioritiser isn't mandatory.
        omimPrioritiser: {},
        #priorityScoreFilter: {minPriorityScore: 0.4},
        #Other prioritisers: Only combine omimPrioritiser with one of these.
        #Don't include any if you only want to filter the variants.
        hiPhivePrioritiser: {},
        # or run hiPhive in benchmarking mode: 
        #hiPhivePrioritiser: {runParams: 'mouse'},
        #phivePrioritiser: {}
        #phenixPrioritiser: {}
        #exomeWalkerPrioritiser: {seedGeneIds: [11111, 22222, 33333]}
    ]
outputOptions:
    outputContributingVariantsOnly: false
    #numGenes options: 0 = all or specify a limit e.g. 500 for the first 500 results  
    numGenes: 0
    # Path to the desired output directory. Will default to the 'results' subdirectory of the exomiser install directory
    outputDirectory: /content/WES_Analysis/Exomiser


    # Filename for the output files. Will default to {input-vcf-filename}-exomiser
    outputFileName: Clinical_Case_01_Exomiser
    #out-format options: HTML, JSON, TSV_GENE, TSV_VARIANT, VCF (default: HTML)
    outputFormats: [HTML, JSON, TSV_GENE, TSV_VARIANT, VCF]

In this case, the analysis uses the annotated VCF prepared in Notebook 2 together with the patient's HPO terms. The results will be written to the specified output directory.

**Exomiser Properties File**

The application.properties file contains global settings used by Exomiser. In most cases, it only needs to be configured once during installation.

For this notebook, the primary purpose of this file is to specify the location of the downloaded Exomiser reference databases. Once the database path has been configured correctly, the same properties file can be reused for all subsequent analyses.

*Unlike the YAML configuration file, the properties file does not change between clinical cases unless the database location or Exomiser installation changes.*

In [ ]:
%%writefile /content/WES_Analysis/Configs/application.properties

## exomiser root data directory ##
exomiser.data-directory=/content/exomiser-cli-14.0.0/data


### hg19 assembly ###
exomiser.hg19.data-version=2406
# transcript source will default to ensembl. Can define as ucsc/ensembl/refseq
exomiser.hg19.transcript-source=ensembl


### phenotypes ###
exomiser.phenotype.data-version=2406
exomiser.phenotype.data-directory=${exomiser.data-directory}/${exomiser.phenotype.data-version}_phenotype


### logging ###
#logging.file.name=logs/exomiser.log
logging.level.com.zaxxer.hikari=ERROR

Writing ./app/My_application.properties


`These directories contain the phenotype, disease, gene, and variant databases required by Exomiser during phenotype-driven prioritization.`

**Checklist**

Before running Exomiser, verify that:

- ✅ The genome assembly matches the annotated VCF.
- ✅ The VCF path is correct.
- ✅ The HPO terms accurately describe the patient's phenotype.
- ✅ The Exomiser database path is correctly configured.
- ✅ The output directory exists.

After confirming that both configuration files are correctly defined, Exomiser can be executed using the YAML configuration file. During the analysis, Exomiser combines genomic variants, patient phenotypes, and curated disease databases to prioritize candidate genes and variants that best explain the clinical presentation.


---